# Eval Overview - Dose Response

As part of our training data, we have chemical perturbations that include different dosages of the same perturbation. During training, we pass the dose into the [ActionComposer](explainer_action_composer_v1_0.ipynb), which uses it to scale the perturbation's action latent. This gives BioJEPA-AC a way to predict different expression changes for the same drug at different concentrations. Because of this, we want to evaluate whether the model has actually learned how a perturbation's impact changes as the dose increases.

To do this evaluation, we built a category of benchmarks called "dose response" that tests whether the model predicts stronger perturbation effects at higher drug concentrations and whether those predictions follow the real response. Since we currently only have one dataset with chemical perturbations, SciPlex, we only analyze that dataset. This analysis uses the same per-sample expression delta we walked through in the [linear expression decoder](explainer_eval_decoders_v1_0.ipynb).

To evaluate dose response, we calculate the monotonicity and Spearman rank correlation for both predicted and real severity across dose levels. We then compare the predicted and real responses by looking at the similarity of each perturbation's dose-response curve and the correlation across perturbations at each dose. As you walk through the notebook, you'll see that we evaluate on multiple dimensions to ensure we have a thorough understanding of where our model is working well and where it's struggling.

In [1]:
import numpy as np
from scipy.stats import pearsonr, spearmanr

In [2]:
SEED = 1337
np.random.seed(SEED)

## Data Prep
We'll start by preparing our data. For this evaluation, we need predicted and real expression deltas, dose values, and perturbation keys for each sample. A "perturbation" is a unique combination of a sequence, target, modality, and mode applied to a cell type. A perturbation can span datasets, though since we're focusing on chemical perturbations only, currently all perturbations are isolated to a single dataset. To show our analysis, we'll need multiple cells that have the same perturbation with different dosages. We'll mock up 3 chemical perturbations across 10 samples to show distinct dose-response behaviors.

For predicted expression, we use the data created by the [linear expression decoder](explainer_eval_decoders_v1_0.ipynb). As a reminder, the decoder takes the BioJEPA-AC latent representation for each gene and uses a linear layer to project each gene down to a single value representing the expression. This gives us an $[\text{n\_genes}]$ expression vector for each sample.

We'll stage our predicted data to show a few different response behaviors: increasing across both dose changes, increasing and then dropping, and decreasing as dose increases. Since our [gene expression prediction explainer notebook](explainer_eval_expr_prediction.ipynb) and others have walked through how sample-level deltas are computed, we'll stage the deltas directly.

In [3]:
num_genes = 8
num_samples = 10
unique_perts = 3

**Perturbations**

We'll first start with our perturbations. Since we're focused on dose response, our perturbations will all be chemical. To define a unique perturbation, it's not just about what we target, but the context of it. Because of this, we represent a unique perturbation as $\text{(seq id, targ id, modality id, mode id, cell type)}$. Notice that the dose is not included in the perturbation key, so samples with the same perturbation at different doses are considered the same perturbation. The dose is still passed separately into the ActionComposer for each sample. Keeping it out of the key allows us to group the same drug together and then compare how its response changes across doses. IDs are used since our model keeps the perturbation information in separate caches from our sample expression counts to avoid heavy duplication of information.

We'll also create a mapping of the samples to the perturbations where the first 4 samples belong to the first perturbation, and the last 6 are split evenly across the two remaining perturbations. We want multiple samples per perturbation to show the effects of different doses.

In [4]:
pert_keys = [
    (0,0,2,4,0),  # Drug A: chemical inhibitor, cell type 0
    (1,1,2,4,0),  # Drug B: chemical inhibitor, cell type 0
    (2,2,2,4,0),  # Drug C: chemical inhibitor, cell type 0
]

sample_to_pert = [0,0,0,0,1,1,1,2,2,2]
sample_to_pert

[0, 0, 0, 0, 1, 1, 1, 2, 2, 2]

**Doses**

Each sample has an associated dose value representing the drug concentration applied to that cell. The raw doses are stored in our training shards so we can pull them out per sample. Before the dose is passed into BioJEPA-AC, we apply a `log1p` transformation:

$$
d = \log(1 + d_{\text{raw}})
$$

This compresses the range of our dose values while keeping their ordering the same. If dose information is unavailable, the value is stored as $-1.0$ and remains $-1.0$ after our data prep. For this evaluation, we use the dose from the first perturbation slot and only keep samples with a positive dose.

For our staged data, we'll use a positional list where each position corresponds to a sample. If we mirror this with our perturbations, you'll see that we have doses of $[0.1, 1, 10]$ for each perturbation, though we have two samples at dose $1.0$ for our first perturbation. We'll keep the staged doses on their original scale so the values are easier to follow. Since `log1p` keeps their ordering the same, this does not change how our monotonicity or rank calculations work.

In [5]:
sample_doses = np.array([0.1,1.0,1.0,10.0,0.1,1.0,10.0,0.1,1.0,10.0])
sample_doses.shape, sample_doses

((10,), array([ 0.1,  1. ,  1. , 10. ,  0.1,  1. , 10. ,  0.1,  1. , 10. ]))

**Predicted Expression Deltas**

To do our evaluation, we also need to know how much the model predicts expression will change for each sample. This prediction is based on the control expression, perturbation, and dose. The dose changes the action latent used by the ACPredictor, while the expression change itself is still calculated as $\hat{\delta}_g=\hat{x}^{\text{case}}_g-\hat{x}^{\text{ctrl}}_g$, the predicted perturbed expression minus the predicted control expression. We use the predicted control expression to isolate BioJEPA-AC's learned perturbation effect from any baseline reconstruction error. Since we have a few different notebooks showing how sample-level deltas are computed, we'll stage the deltas directly.

We've staged our predicted expression deltas to show three behaviors: Drug A increases across both dose changes, Drug B increases at the middle dose and then drops, and Drug C decreases as dose increases. You'll see how these different behaviors flow through each of our dose-response calculations.

In [6]:
pred_deltas = np.array([
    [0.1,-0.2,0.1,0.0,0.1,0.0,-0.1,0.0],        # Drug A, dose 0.1
    [0.4,-0.5,0.3,0.2,0.3,-0.1,-0.4,0.1],        # Drug A, dose 1.0
    [0.5,-0.4,0.4,0.1,0.2,-0.2,-0.3,0.2],        # Drug A, dose 1.0
    [1.0,-0.8,0.7,0.5,0.6,-0.3,-0.9,0.4],        # Drug A, dose 10.0
    [0.05,-0.1,0.05,0.1,0.0,0.0,-0.05,0.0],      # Drug B, dose 0.1
    [0.8,-0.7,0.5,0.4,0.3,-0.2,-0.6,0.3],        # Drug B, dose 1.0
    [0.5,-0.3,0.3,0.2,0.2,-0.1,-0.4,0.2],        # Drug B, dose 10.0
    [0.3,-0.3,0.2,0.1,0.2,-0.1,-0.2,0.1],        # Drug C, dose 0.1
    [0.3,-0.2,0.25,0.05,0.15,-0.15,-0.25,0.15],  # Drug C, dose 1.0
    [0.25,-0.25,0.15,0.15,0.1,-0.1,-0.15,0.15],  # Drug C, dose 10.0
])
pred_deltas.shape, pred_deltas

((10, 8),
 array([[ 0.1 , -0.2 ,  0.1 ,  0.  ,  0.1 ,  0.  , -0.1 ,  0.  ],
        [ 0.4 , -0.5 ,  0.3 ,  0.2 ,  0.3 , -0.1 , -0.4 ,  0.1 ],
        [ 0.5 , -0.4 ,  0.4 ,  0.1 ,  0.2 , -0.2 , -0.3 ,  0.2 ],
        [ 1.  , -0.8 ,  0.7 ,  0.5 ,  0.6 , -0.3 , -0.9 ,  0.4 ],
        [ 0.05, -0.1 ,  0.05,  0.1 ,  0.  ,  0.  , -0.05,  0.  ],
        [ 0.8 , -0.7 ,  0.5 ,  0.4 ,  0.3 , -0.2 , -0.6 ,  0.3 ],
        [ 0.5 , -0.3 ,  0.3 ,  0.2 ,  0.2 , -0.1 , -0.4 ,  0.2 ],
        [ 0.3 , -0.3 ,  0.2 ,  0.1 ,  0.2 , -0.1 , -0.2 ,  0.1 ],
        [ 0.3 , -0.2 ,  0.25,  0.05,  0.15, -0.15, -0.25,  0.15],
        [ 0.25, -0.25,  0.15,  0.15,  0.1 , -0.1 , -0.15,  0.15]]))

**Real Expression Deltas**

Next we'll stage the real expression deltas. We calculate the real expression change as $\delta_g = x^{\text{case}}_g - x^{\text{ctrl}}_g$, the real perturbed expression minus the real control expression. This gives us the real dose-response behavior that we'll compare against our predictions.

We've staged the real deltas separately from our predicted deltas so the two curves are not identical. The real severity increases with dose for all three drugs, while our predictions capture that behavior well for Drug A, partially for Drug B, and poorly for Drug C.

In [7]:
real_deltas = np.array([
    [0.1,-0.2,0.1,0.1,0.1,0.0,-0.1,0.0],        # Drug A, dose 0.1
    [0.5,-0.5,0.4,0.2,0.3,-0.2,-0.4,0.2],        # Drug A, dose 1.0
    [0.5,-0.6,0.3,0.3,0.4,-0.1,-0.5,0.1],        # Drug A, dose 1.0
    [1.1,-0.9,0.8,0.6,0.7,-0.4,-1.0,0.5],        # Drug A, dose 10.0
    [0.1,-0.1,0.1,0.0,0.1,0.0,-0.1,0.0],         # Drug B, dose 0.1
    [0.4,-0.4,0.3,0.2,0.2,-0.1,-0.3,0.2],        # Drug B, dose 1.0
    [0.9,-0.8,0.7,0.5,0.6,-0.3,-0.8,0.4],        # Drug B, dose 10.0
    [0.1,-0.2,0.1,0.1,0.1,0.0,-0.1,0.0],         # Drug C, dose 0.1
    [0.4,-0.4,0.3,0.2,0.3,-0.1,-0.3,0.2],        # Drug C, dose 1.0
    [0.8,-0.7,0.6,0.4,0.5,-0.3,-0.7,0.4],        # Drug C, dose 10.0
])
real_deltas.shape, real_deltas

((10, 8),
 array([[ 0.1, -0.2,  0.1,  0.1,  0.1,  0. , -0.1,  0. ],
        [ 0.5, -0.5,  0.4,  0.2,  0.3, -0.2, -0.4,  0.2],
        [ 0.5, -0.6,  0.3,  0.3,  0.4, -0.1, -0.5,  0.1],
        [ 1.1, -0.9,  0.8,  0.6,  0.7, -0.4, -1. ,  0.5],
        [ 0.1, -0.1,  0.1,  0. ,  0.1,  0. , -0.1,  0. ],
        [ 0.4, -0.4,  0.3,  0.2,  0.2, -0.1, -0.3,  0.2],
        [ 0.9, -0.8,  0.7,  0.5,  0.6, -0.3, -0.8,  0.4],
        [ 0.1, -0.2,  0.1,  0.1,  0.1,  0. , -0.1,  0. ],
        [ 0.4, -0.4,  0.3,  0.2,  0.3, -0.1, -0.3,  0.2],
        [ 0.8, -0.7,  0.6,  0.4,  0.5, -0.3, -0.7,  0.4]]))

## Dose-Response Evaluation

Now we'll evaluate whether our predicted and real expression changes scale with dose. To do this, we first convert our expression deltas into scalar severity scores, take the per-perturbation and dose averages, and then evaluate how each changes with dosage. Finally, we'll compare our predicted and real dose-response behavior.

### Severity Calculation

We'll start by calculating the severity per sample for both our predicted and real expression deltas. We calculate severity as the L2 norm of the expression delta vector. This collapses the per-gene changes into a single number representing the overall magnitude of the perturbation effect. We calculate both as:
$$\begin{aligned}
\hat{s}_s &= \|\hat{\delta}_s\| = \sqrt{\sum_{g=1}^{G}\hat{\delta}_{s,g}^{2}} \\
s_s &= \|\delta_s\| = \sqrt{\sum_{g=1}^{G}\delta_{s,g}^{2}}
\end{aligned}$$

Since all 8 genes in our staged data are measured, we'll use every gene in our calculation. In our production evaluation, we calculate severity using only the genes measured in the SciPlex dataset.

Because of how we staged our data, you'll see the predicted and real severity differ across the samples. This difference is what will let us evaluate whether the predicted curves follow the real dose-response curves.

In [8]:
pred_severity = np.linalg.norm(pred_deltas, axis=1)
pred_severity.shape, pred_severity

((10,),
 array([0.28284271, 0.9       , 0.88881944, 1.94935887, 0.16583124,
        1.45602198, 0.84852814, 0.57445626, 0.57008771, 0.48476799]))

In [9]:
real_severity = np.linalg.norm(real_deltas, axis=1)
real_severity.shape, real_severity

((10,),
 array([0.3       , 1.01488916, 1.1045361 , 2.2181073 , 0.2236068 ,
        0.79372539, 1.8547237 , 0.3       , 0.82462113, 1.62480768]))

**Per-Perturbation & Dose Average**

Now that we have predicted and real severity per sample, we need to group the samples by drug and dose level. Using our `sample_to_pert` mapping, we can identify which drug each sample received, then pair it with the dose and both severity values.

Within each drug, we group by dose level and average the predicted and real severity across replicates. We calculate these means as:
$$\begin{aligned}
\bar{\hat{s}}_{p,d} &= \frac{1}{|R_{p,d}|}\sum_{s \in R_{p,d}}\hat{s}_s \\
\bar{s}_{p,d} &= \frac{1}{|R_{p,d}|}\sum_{s \in R_{p,d}}s_s
\end{aligned}$$

where $R_{p,d}$ is the set of samples for drug $p$ at dose level $d$. For the first perturbation, we have two samples at dose 1.0 that will be averaged into a single predicted severity and a single real severity.

In [10]:
pert_dose_pred_sev = {}
pert_dose_real_sev = {}

In [11]:
for i in range(num_samples):
    print(f'---- sample {i} ----')
    pert_idx = sample_to_pert[i]
    dosage = sample_doses[i]
    pred_sev = pred_severity[i]
    real_sev = real_severity[i]
    print(f'pert_id {pert_idx} | dosage {dosage} | pred sev {pred_sev} | real sev {real_sev}')

    if pert_idx not in pert_dose_pred_sev:
        pert_dose_pred_sev[pert_idx] = {}
        pert_dose_real_sev[pert_idx] = {}
    if dosage not in pert_dose_pred_sev[pert_idx]:
        pert_dose_pred_sev[pert_idx][dosage] = []
        pert_dose_real_sev[pert_idx][dosage] = []

    pert_dose_pred_sev[pert_idx][dosage].append(pred_sev)
    pert_dose_real_sev[pert_idx][dosage].append(real_sev)

---- sample 0 ----
pert_id 0 | dosage 0.1 | pred sev 0.28284271247461906 | real sev 0.30000000000000004
---- sample 1 ----
pert_id 0 | dosage 1.0 | pred sev 0.9 | real sev 1.0148891565092222
---- sample 2 ----
pert_id 0 | dosage 1.0 | pred sev 0.8888194417315589 | real sev 1.104536101718726
---- sample 3 ----
pert_id 0 | dosage 10.0 | pred sev 1.9493588689617927 | real sev 2.2181073012818833
---- sample 4 ----
pert_id 1 | dosage 0.1 | pred sev 0.16583123951777 | real sev 0.223606797749979
---- sample 5 ----
pert_id 1 | dosage 1.0 | pred sev 1.4560219778561037 | real sev 0.7937253933193773
---- sample 6 ----
pert_id 1 | dosage 10.0 | pred sev 0.848528137423857 | real sev 1.854723699099141
---- sample 7 ----
pert_id 2 | dosage 0.1 | pred sev 0.5744562646538028 | real sev 0.30000000000000004
---- sample 8 ----
pert_id 2 | dosage 1.0 | pred sev 0.570087712549569 | real sev 0.8246211251235321
---- sample 9 ----
pert_id 2 | dosage 10.0 | pred sev 0.4847679857416329 | real sev 1.6248076809271

In [12]:
pert_dose_pred_sev

{0: {np.float64(0.1): [np.float64(0.28284271247461906)],
  np.float64(1.0): [np.float64(0.9), np.float64(0.8888194417315589)],
  np.float64(10.0): [np.float64(1.9493588689617927)]},
 1: {np.float64(0.1): [np.float64(0.16583123951777)],
  np.float64(1.0): [np.float64(1.4560219778561037)],
  np.float64(10.0): [np.float64(0.848528137423857)]},
 2: {np.float64(0.1): [np.float64(0.5744562646538028)],
  np.float64(1.0): [np.float64(0.570087712549569)],
  np.float64(10.0): [np.float64(0.4847679857416329)]}}

In [13]:
pert_dose_real_sev

{0: {np.float64(0.1): [np.float64(0.30000000000000004)],
  np.float64(1.0): [np.float64(1.0148891565092222),
   np.float64(1.104536101718726)],
  np.float64(10.0): [np.float64(2.2181073012818833)]},
 1: {np.float64(0.1): [np.float64(0.223606797749979)],
  np.float64(1.0): [np.float64(0.7937253933193773)],
  np.float64(10.0): [np.float64(1.854723699099141)]},
 2: {np.float64(0.1): [np.float64(0.30000000000000004)],
  np.float64(1.0): [np.float64(0.8246211251235321)],
  np.float64(10.0): [np.float64(1.624807680927192)]}}

**Mean per Perturbation & Dose**

Now we'll calculate the predicted and real mean severity at each dose. For our predicted severity, the first perturbation increases with dose, the second peaks at the middle dose, and the third decreases. Our real severity increases with dose for all three perturbations.

In [14]:
avg_pred_sev = {p: {d: np.mean(sevs) for d, sevs in doses.items()} for p, doses in pert_dose_pred_sev.items()}
avg_pred_sev

{0: {np.float64(0.1): np.float64(0.28284271247461906),
  np.float64(1.0): np.float64(0.8944097208657795),
  np.float64(10.0): np.float64(1.9493588689617927)},
 1: {np.float64(0.1): np.float64(0.16583123951777),
  np.float64(1.0): np.float64(1.4560219778561037),
  np.float64(10.0): np.float64(0.848528137423857)},
 2: {np.float64(0.1): np.float64(0.5744562646538028),
  np.float64(1.0): np.float64(0.570087712549569),
  np.float64(10.0): np.float64(0.4847679857416329)}}

In [15]:
avg_real_sev = {p: {d: np.mean(sevs) for d, sevs in doses.items()} for p, doses in pert_dose_real_sev.items()}
avg_real_sev

{0: {np.float64(0.1): np.float64(0.30000000000000004),
  np.float64(1.0): np.float64(1.059712629113974),
  np.float64(10.0): np.float64(2.2181073012818833)},
 1: {np.float64(0.1): np.float64(0.223606797749979),
  np.float64(1.0): np.float64(0.7937253933193773),
  np.float64(10.0): np.float64(1.854723699099141)},
 2: {np.float64(0.1): np.float64(0.30000000000000004),
  np.float64(1.0): np.float64(0.8246211251235321),
  np.float64(10.0): np.float64(1.624807680927192)}}

### Monotonicity Score

Now we're ready to evaluate what fraction of consecutive dose increases actually produce an increase in severity. This helps us understand how consistently severity rises as dosage increases. We calculate the score separately for our predicted and real severity curves:
$$\begin{aligned}
\text{Monotonicity}_{\text{pred}} &= \frac{1}{T}\sum_{t=1}^{T}\mathbf{1}[\bar{\hat{s}}_{p,d_{t+1}} > \bar{\hat{s}}_{p,d_t}] \\
\text{Monotonicity}_{\text{real}} &= \frac{1}{T}\sum_{t=1}^{T}\mathbf{1}[\bar{s}_{p,d_{t+1}} > \bar{s}_{p,d_t}]
\end{aligned}$$

where $T$ is the total number of consecutive dose-level pairs across all drugs. The predicted score tells us how often BioJEPA-AC predicts increasing severity, while the real score tells us how often severity actually increases in the dataset. Comparing the two helps us understand whether poor predicted monotonicity reflects the model or the underlying response.

Because of how we staged the data, our first predicted perturbation will show perfect monotonicity, our second will only show it for the first dosage change but not the second, and our third, since it decreases, will show 0. Our real severity will show perfect monotonicity for all three perturbations.

In [16]:
total_pred_mon_count = 0.0
total_real_mon_count = 0.0
total_pairs = 0.0

In [17]:
for p in avg_pred_sev:
    print(f'---- pert {p} ----')
    sorted_doses = sorted(avg_pred_sev[p].keys())
    pred_sev = [avg_pred_sev[p][d] for d in sorted_doses]
    real_sev = [avg_real_sev[p][d] for d in sorted_doses]
    print(f'pred {pred_sev}')
    print(f'real {real_sev}')

    pred_mono_counts = sum(1 for i in range(len(pred_sev) - 1) if pred_sev[i + 1] > pred_sev[i])
    real_mono_counts = sum(1 for i in range(len(real_sev) - 1) if real_sev[i + 1] > real_sev[i])
    pairs = len(sorted_doses) - 1

    total_pred_mon_count += pred_mono_counts
    total_real_mon_count += real_mono_counts
    total_pairs += pairs

    print(f'pred monotonicity {pred_mono_counts / pairs} | real monotonicity {real_mono_counts / pairs}')

---- pert 0 ----
pred [np.float64(0.28284271247461906), np.float64(0.8944097208657795), np.float64(1.9493588689617927)]
real [np.float64(0.30000000000000004), np.float64(1.059712629113974), np.float64(2.2181073012818833)]
pred monotonicity 1.0 | real monotonicity 1.0
---- pert 1 ----
pred [np.float64(0.16583123951777), np.float64(1.4560219778561037), np.float64(0.848528137423857)]
real [np.float64(0.223606797749979), np.float64(0.7937253933193773), np.float64(1.854723699099141)]
pred monotonicity 0.5 | real monotonicity 1.0
---- pert 2 ----
pred [np.float64(0.5744562646538028), np.float64(0.570087712549569), np.float64(0.4847679857416329)]
real [np.float64(0.30000000000000004), np.float64(0.8246211251235321), np.float64(1.624807680927192)]
pred monotonicity 0.0 | real monotonicity 1.0


**Overall Monotonicity**

Now we're ready to get our overall predicted and real monotonicity scores. If you look at our loop, these are actually pair-weighted means, so if one perturbation has more dose levels than the others, it will contribute more.

In [18]:
monotonicity_score = float(total_pred_mon_count / total_pairs)
monotonicity_score

0.5

In [19]:
real_monotonicity_score = float(total_real_mon_count / total_pairs)
real_monotonicity_score

1.0

### Spearman Rank Correlation

Monotonicity gives us a count of how often severity increases when dose increases, but we also want to know if, across our dataset, severity is correlated with dose. We use Spearman correlation to measure whether the rank ordering of doses matches the rank ordering of mean severities. We calculate it as:
$$\begin{aligned}
\rho_{\text{pred}} &= \operatorname{corr}(R(d), R(\bar{\hat{s}})) \\
\rho_{\text{real}} &= \operatorname{corr}(R(d), R(\bar{s}))
\end{aligned}$$
where $R(d)$, $R(\bar{\hat{s}})$, and $R(\bar{s})$ are the ranks of the dose, predicted mean severity, and real mean severity values. Since the same dose appears across multiple perturbations, we use Spearman's correlation so those repeated dose values are handled as tied ranks. We compute this across all dose-level means from all drugs pooled together, so each perturbation contributes its own set of (dose, severity) points. Because of how we staged our data, we'll see a moderate positive correlation for our predicted severity and a stronger positive correlation for our real severity.

In [20]:
all_doses_flat = [d for p in avg_pred_sev for d in sorted(avg_pred_sev[p].keys())]
all_doses_flat

[np.float64(0.1),
 np.float64(1.0),
 np.float64(10.0),
 np.float64(0.1),
 np.float64(1.0),
 np.float64(10.0),
 np.float64(0.1),
 np.float64(1.0),
 np.float64(10.0)]

In [21]:
all_pred_severities_flat = [avg_pred_sev[p][d] for p in avg_pred_sev for d in sorted(avg_pred_sev[p].keys())]
all_pred_severities_flat

[np.float64(0.28284271247461906),
 np.float64(0.8944097208657795),
 np.float64(1.9493588689617927),
 np.float64(0.16583123951777),
 np.float64(1.4560219778561037),
 np.float64(0.848528137423857),
 np.float64(0.5744562646538028),
 np.float64(0.570087712549569),
 np.float64(0.4847679857416329)]

In [22]:
all_real_severities_flat = [avg_real_sev[p][d] for p in avg_real_sev for d in sorted(avg_real_sev[p].keys())]
all_real_severities_flat

[np.float64(0.30000000000000004),
 np.float64(1.059712629113974),
 np.float64(2.2181073012818833),
 np.float64(0.223606797749979),
 np.float64(0.7937253933193773),
 np.float64(1.854723699099141),
 np.float64(0.30000000000000004),
 np.float64(0.8246211251235321),
 np.float64(1.624807680927192)]

In [23]:
pred_r, _ = spearmanr(all_doses_flat, all_pred_severities_flat)
dose_severity_spearman = 0.0 if np.isnan(pred_r) else float(pred_r)
dose_severity_spearman

0.5270462766947299

In [24]:
real_r, _ = spearmanr(all_doses_flat, all_real_severities_flat)
real_dose_severity_spearman = 0.0 if np.isnan(real_r) else float(real_r)
real_dose_severity_spearman

0.9526610232449337

### Curve Similarity

Our monotonicity and Spearman calculations tell us whether the predicted and real severity generally increase with dose, but they do not directly compare the shape of the predicted and real curves. For this comparison, we take the predicted and real mean severity across dose levels for each perturbation and calculate Pearson's $r$. We then take the mean of those correlations across perturbations:

$$\begin{aligned}
r_p &= \operatorname{corr}_{d \in D_p}(\bar{\hat{s}}_{p,d}, \bar{s}_{p,d}) \\
\text{Curve Similarity} &= \frac{1}{|P'|}\sum_{p \in P'}r_p
\end{aligned}$$

where $D_p$ is the set of dose levels for perturbation $p$, and $P'$ is the set of perturbations where both the predicted and real severity change across dose levels. If either curve is flat, Pearson's $r$ is not meaningful, so we leave that perturbation out of the production average.

A higher curve similarity means the predicted severity changes across dose levels in a similar way to the real severity. Because of how we staged our data, Drug A will have a strong positive correlation, Drug B will have a weaker correlation, and Drug C will have a negative correlation.

In [25]:
curve_similarities = []

In [26]:
for p in avg_pred_sev:
    print(f'---- pert {p} ----')
    sorted_doses = sorted(avg_pred_sev[p].keys())
    pred_curve = [avg_pred_sev[p][d] for d in sorted_doses]
    real_curve = [avg_real_sev[p][d] for d in sorted_doses]

    curve_r, _ = pearsonr(pred_curve, real_curve)
    curve_r = 0.0 if np.isnan(curve_r) else float(curve_r)
    print(f'curve similarity {curve_r}')
    curve_similarities.append(curve_r)

---- pert 0 ----
curve similarity 0.9994561265962307
---- pert 1 ----
curve similarity 0.37574869532325267
---- pert 2 ----
curve similarity -0.9355895117754955


In [27]:
curve_similarity = float(np.mean(curve_similarities))
curve_similarity

0.14653843671466263

### Predicted vs. Real Severity by Dose

Our curve similarity looks at one perturbation across all its doses. Now we'll flip that analysis and look at one dose across all our perturbations. This tells us whether the perturbations predicted to have the strongest effects are also the perturbations with the strongest real effects at that dose. We calculate it as:

$$
r_d = \operatorname{corr}_{p \in P_d}(\bar{\hat{s}}_{p,d}, \bar{s}_{p,d})
$$

where $P_d$ is the set of perturbations measured at dose $d$. For each dose, we collect the mean predicted and real severity for every perturbation and calculate Pearson's $r$ across those values. In our production evaluation, we only calculate the correlation when a dose has at least 3 perturbations and both sets of severity values vary.

Because we staged 3 perturbations at each dose, we'll be able to calculate the correlation for all three dose levels. You'll see that the model does a better job ordering the perturbations at some doses than others.

In [28]:
pred_vs_real_by_dose = {}

In [29]:
for dose in sorted(set(all_doses_flat)):
    print(f'---- dose {dose} ----')
    pred_at_dose = [avg_pred_sev[p][dose] for p in avg_pred_sev if dose in avg_pred_sev[p]]
    real_at_dose = [avg_real_sev[p][dose] for p in avg_real_sev if dose in avg_real_sev[p]]
    print(f'pred {pred_at_dose}')
    print(f'real {real_at_dose}')

    dose_result = {'n_drugs': len(pred_at_dose), 'pearson': None}
    if len(pred_at_dose) >= 3 and np.std(pred_at_dose) > 1e-9 and np.std(real_at_dose) > 1e-9:
        dose_r, _ = pearsonr(pred_at_dose, real_at_dose)
        dose_result['pearson'] = 0.0 if np.isnan(dose_r) else float(dose_r)

    pred_vs_real_by_dose[dose] = dose_result

---- dose 0.1 ----
pred [np.float64(0.28284271247461906), np.float64(0.16583123951777), np.float64(0.5744562646538028)]
real [np.float64(0.30000000000000004), np.float64(0.223606797749979), np.float64(0.30000000000000004)]
---- dose 1.0 ----
pred [np.float64(0.8944097208657795), np.float64(1.4560219778561037), np.float64(0.570087712549569)]
real [np.float64(1.059712629113974), np.float64(0.7937253933193773), np.float64(0.8246211251235321)]
---- dose 10.0 ----
pred [np.float64(1.9493588689617927), np.float64(0.848528137423857), np.float64(0.4847679857416329)]
real [np.float64(2.2181073012818833), np.float64(1.854723699099141), np.float64(1.624807680927192)]


In [30]:
pred_vs_real_by_dose

{np.float64(0.1): {'n_drugs': 3, 'pearson': 0.7210593483053782},
 np.float64(1.0): {'n_drugs': 3, 'pearson': -0.25690234466522666},
 np.float64(10.0): {'n_drugs': 3, 'pearson': 0.9882244045699395}}

## Dose Response Final Wrapup

We've now walked through the dose response evaluation that tests whether a model's predicted severity scales with perturbation dose. We calculated how the predicted and real severity change across dose levels, then compared the predicted and real curves both within each perturbation and across perturbations at each dose. Together, these calculations help us understand whether BioJEPA-AC has learned the strength and shape of the real dose response.